In [5]:
import pandas as pd
import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


# Predict biogas emitions daily to optimize the plants capacity

## 1. Formulate the research problem

Biogas production from organic waste plays an important role in sustainable energy generation and waste management. The ability to accurately predict daily biogas output enables better planning of energy supply and optimization of the anaerobic digestion process. This project uses a 15-year representing dataset from a biogas plant in India to develop regression models capable of predicting daily biogas production and supporting operational decision-making in similar facilities.

### 1.1 Business Perspective
Can daily biogas production be predicted accurately from waste composition, operational parameters, and weather conditions?

### 1.2 Data science Perspective
Build regression model that predicts continous value of Raw Biogas Produced (m³) using 25 scalar values. Model will be evaluated base on R² and RMSE where a good values will be considered R^2 > 0.75 and RMSE < 10% for mean biogas production.
Why these values? 0.75 will show a good 

For a biogas plant prediction of 10% error is acceptable because the gas storage buffer absorbs the daily fluctuations. According to the literature gasholders are designed as buffer devices that adjust peaks and fluctuations in biogas production.<sup>[1]</sup> The typical storage production is 10-30% of the daily production for continuously operating systems. A prediction error of 10% falls into this range, which means that the plant can absorb such a deviation without operation disturbance.    

## 2. Hypothesis
Null hypothesis (H₀)

The available input variables (waste composition, operational parameters, and weather conditions) do not provide meaningful predictive information about daily biogas production.

Any prediction model will perform no better than a simple baseline using the average daily production.

Feedstock quantities, biological process parameters, and environmental conditions are not associated with daily biogas production.

Alternative hypothesis (H₁)

Feedstock quantities, biological process parameters, and environmental conditions are not associated with daily biogas production. Specifically, higher quantities of organic feedstock and favorable digestion conditions are expected to increase biogas production.

A regression model trained on these variables will significantly outperform a baseline model.

## 2. Hypothesis

A machine learning model can predict daily biogas production from feedstock composition, environmental conditions, and operational parameters with useful accuracy (R² > 0.75).


## 3. Expected Outcomes

Based on domain knowledge of anaerobic digestion, waste-related variables are expected to be among the most important predictors of biogas production.

We expect:

* Models using waste composition, operational parameters, and weather conditions to outperform a baseline model.
* Waste quantity variables to show a positive relationship with biogas production.
* Feature importance analysis to identify waste composition as one of the key drivers of production.

The hypothesis will be considered supported if:

* the final model achieves R² ≥ 0.75;
* RMSE remains below 10% of the mean daily production;
* models including waste-related features perform substantially better than models using weather variables alone.

Summary:
We expect tree-based ensemble models (Random Forest and Gradient Boosting) to outperform a simple linear regression baseline because biogas production is influenced by multiple interacting process and environmental variables.

## Exploratory Data Analysis
* What data do I have?

*Dataset*

The dataset contains 25 numerical features describing feedstock composition, operational parameters, environmental conditions, and biological process characteristics. Examples include pig manure, kitchen food waste, volatile solids, digester temperature, and hydraulic retention time (HRT). The target variable is daily raw biogas production measured in cubic meters (m³).

Explaination of the features

Based on domain knowledge of anaerobic digestion, the variables expected to have the strongest influence on biogas production are:

* Kitchen food waste
* Pig manure
* Chicken litter
* Volatile solids
* Digester temperature
* Hydraulic retention time (HRT)
* C/N ratio

*Technical Background*

Several variables in the dataset are known to influence biogas production.

**Feedstock Composition**

Biogas is produced through anaerobic digestion of organic materials. Feedstocks such as kitchen food waste, pig manure, chicken litter, and municipal residues provide the organic matter that microorganisms convert into biogas. In general, larger quantities of biodegradable material are expected to increase gas production.

**Volatile Solids (VS)**

Volatile solids represent the biodegradable portion of the feedstock. Since microorganisms consume this organic matter during digestion, higher volatile solids content is generally associated with greater biogas production potential.

**Digester Temperature**

Microbial activity depends strongly on temperature. Digestion is most efficient within specific temperature ranges, while lower temperatures can reduce gas production by slowing microbial processes.

**Hydraulic Retention Time (HRT)**

Hydraulic Retention Time measures how long the feedstock remains inside the digester. Longer retention times allow microorganisms more time to break down organic material and produce biogas.

**Carbon-to-Nitrogen Ratio (C/N Ratio)**

The C/N ratio describes the balance between carbon and nitrogen in the feedstock. An appropriate balance supports microbial growth and efficient digestion, while an imbalance may reduce biogas yield.

* Are there missing values?
  


In [7]:
biogas_data = pd.read_csv('biogas_dataset.csv')
biogas_data.head()

,Year,Month,Day,Pig Manure (kg),Kitchen Food Waste (kg),Chicken Litter (kg),Cassava (kg),Bagasse Feed (kg),Energy Grass (kg),Banana Shafts (kg),...,Fish Waste (kg),Water (L),Diesel (L),Electricity Use (kWh),Temperature (C),Humidity (%),Rainfall (mm),C/N Ratio,Digester Temp (C),biogas_production
0,2010,1,1,14.785537,7.305310,13.050192,17.684926,14.830197,9.906063,7.743000,...,4.191195,82.619547,2.866209,25.258583,33.135037,75.997637,10.193971,26.666653,36.841906,58.956420
1,2010,1,1,15.254357,16.039052,12.686489,10.436417,17.292921,16.466707,5.785339,...,5.903021,107.321735,3.588735,34.322305,29.268128,76.987494,1.284466,23.100934,32.535921,71.951004
2,2010,1,1,35.566142,19.992287,8.336992,33.463391,15.889184,5.330412,9.085270,...,4.300494,138.387090,0.070003,18.190760,31.736523,81.425018,3.108892,21.273712,37.357393,102.965090
3,2010,1,2,34.496363,17.751484,8.232446,14.096671,14.088245,8.093446,11.882885,...,7.469573,99.479363,1.532482,44.195734,29.677593,85.800528,1.940286,31.250011,33.820132,83.783745
4,2010,1,2,21.187255,13.533476,14.489672,14.756190,12.165611,12.064431,4.597625,...,2.617079,86.328262,0.362056,30.257209,29.731208,73.236587,2.442031,20.177435,36.571963,70.489410


Lets look at default data types:

In [9]:
biogas_data.dtypes

Year                         int64
Month                        int64
Day                          int64
Pig Manure (kg)            float64
Kitchen Food Waste (kg)    float64
Chicken Litter (kg)        float64
Cassava (kg)               float64
Bagasse Feed (kg)          float64
Energy Grass (kg)          float64
Banana Shafts (kg)         float64
Alcohol Waste (kg)         float64
Municipal Residue (kg)     float64
Fish Waste (kg)            float64
Water (L)                  float64
Diesel (L)                 float64
Electricity Use (kWh)      float64
Temperature (C)            float64
Humidity (%)               float64
Rainfall (mm)              float64
C/N Ratio                  float64
Digester Temp (C)          float64
biogas_production          float64
dtype: object

In [ ]:
As we can see, all features are numeric which is a good base for modeling.

In [10]:
biogas_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 15298 entries, 0 to 15297
Data columns (total 22 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Year                     15298 non-null  int64  
 1   Month                    15298 non-null  int64  
 2   Day                      15298 non-null  int64  
 3   Pig Manure (kg)          15298 non-null  float64
 4   Kitchen Food Waste (kg)  15298 non-null  float64
 5   Chicken Litter (kg)      15298 non-null  float64
 6   Cassava (kg)             15298 non-null  float64
 7   Bagasse Feed (kg)        15298 non-null  float64
 8   Energy Grass (kg)        15298 non-null  float64
 9   Banana Shafts (kg)       15298 non-null  float64
 10  Alcohol Waste (kg)       15298 non-null  float64
 11  Municipal Residue (kg)   15298 non-null  float64
 12  Fish Waste (kg)          15298 non-null  float64
 13  Water (L)                15298 non-null  float64
 14  Diesel (L)               15298 no

This means we can ignore encoding completely in the preprocessing phase.
Check for missing values

In [12]:
print("Missing values in every column : \n", biogas_data.isnull().sum())
print("\n\n Missing values total : ", biogas_data.isnull().sum().sum())

Missing values in every column : 
 Year                       0
Month                      0
Day                        0
Pig Manure (kg)            0
Kitchen Food Waste (kg)    0
Chicken Litter (kg)        0
Cassava (kg)               0
Bagasse Feed (kg)          0
Energy Grass (kg)          0
Banana Shafts (kg)         0
Alcohol Waste (kg)         0
Municipal Residue (kg)     0
Fish Waste (kg)            0
Water (L)                  0
Diesel (L)                 0
Electricity Use (kWh)      0
Temperature (C)            0
Humidity (%)               0
Rainfall (mm)              0
C/N Ratio                  0
Digester Temp (C)          0
biogas_production          0
dtype: int64


 Missing values total :  0


Check for dublicates

In [13]:
print(df.columns)

Index(['Year', 'Month', 'Day', 'Pig Manure (kg)', 'Kitchen Food Waste (kg)',
       'Chicken Litter (kg)', 'Cassava (kg)', 'Bagasse Feed (kg)',
       'Energy Grass (kg)', 'Banana Shafts (kg)', 'Alcohol Waste (kg)',
       'Municipal Residue (kg)', 'Fish Waste (kg)', 'Water (L)', 'Diesel (L)',
       'Electricity Use (kWh)', 'Temperature (C)', 'Humidity (%)',
       'Rainfall (mm)', 'C/N Ratio', 'Digester Temp (C)', 'biogas_production'],
      dtype='str')


In [16]:
df.duplicated().sum()

np.int64(0)

### Data Cleaning

Before building the machine learning models, the dataset was inspected for common data quality issues.

*Missing Values*

The dataset was checked for missing values using df.isnull().sum(). No missing values were found, therefore no imputation techniques were required.

*Duplicate Records*

The dataset was examined for duplicate observations using df.duplicated().sum(). No duplicate records were identified and no rows were removed.

*Data Types*

All variables were stored as numerical values (int64 or float64), which are suitable for regression models. No categorical features were present, therefore no encoding techniques such as one-hot encoding or label encoding were required.

*Data Consistency*

The feature values were inspected using summary statistics (df.describe()) to verify that the variables were within realistic operational ranges. No obvious data quality issues were identified during the initial inspection.

As a result, the dataset required minimal preprocessing and was suitable for exploratory data analysis and model development.

* What ranges do the variables have?
* Are there outliers?
* Which variables seem related to the target?
* Are some variables highly correlated?

5. Exploratory Data Analysis
6. Models
7. Evaluation
8. Conclusion (Was the hypothesis supported?)
  
## 4. Preprocessing so models can be fittd

## 5. Models to Test

Linear Regression, Polynomial, Ridge/Lasso, Decision Tree, Random Forest

## 6. Linear Regression - evaluation (слаб модел няма голям капацитет, подаваш му много данни но няма способност да ги научи overfitting)

## 7. Try another model - да се подкара друг модел - оценка


## 8. Регулация на линейната регресия

## 9.
## 10. Една значителна промяна, обработват се по-различен начин, начина на подаване на данните да е измеримо различен, данните не се променят

## добре е да не се променят оценките за да може да се сравни

### References

**Deng, L., Liu, Y., & Wang, W. (2020).** Biogas Storage. In *Biogas Technology*. Springer.  
*Note:* Provides foundational knowledge on biogas system components.

**University of Southampton. (2015).** *The effect of C/N ratio on anaerobic digestion of kitchen food waste and chicken litter*. ePrints Soton. https://eprints.soton.ac.uk/381551/  
*Note:* Best for understanding ammonia toxicity and co-digestion challenges with high-nitrogen feedstocks.